In [11]:
import os
from uuid import uuid4

from dotenv import load_dotenv
from iaragenai import IaraGenAI

load_dotenv()

print(f"Client ID: {os.getenv('IARA_CLIENT_ID')}")
print(f"Environment: {os.getenv('IARA_ENVIRONMENT')}")
def main():
    client = IaraGenAI(
        client_id=os.getenv("IARA_CLIENT_ID", "your_client_id"),
        client_secret=os.getenv("IARA_CLIENT_SECRET", "your_client_secret"),
        environment=os.getenv("IARA_ENVIRONMENT", "prod"),
        access_token=os.getenv("IARA_ACCESS_TOKEN", "your_project_key_access"),
        correlation_id=str(uuid4()),
    )

    kbs = client.knowledge_base.list()

    print(kbs)


if __name__ == "__main__":
    main()

Client ID: b99d744d-1a6c-4d87-a5d4-e9349bf9cfb5
Environment: homol
[KnowledgeBaseResponse(knowledge_base_id='112dc826-d2b7-4e70-9573-4420ab000bc0', project_id='95de11d3-a7f6-4ab1-bc61-59180e62b0b4', user_id='0b9b68df-88dc-4401-b0cb-fbc967aea677', name='KB_Catchs_Auditoria', description='KB de catchs da auditoria. Criado para homologação do agente do Multiagente RO', access_level=<AccessLevelType.PROTECTED: 'PROTECTED'>, embedding_model=EmbeddingModel(provider_name='azure_openai', model_name='text-embedding-ada-002', model_dimension=1536), created_at='2026-03-27T20:36:26Z', updated_at='2026-03-27T20:36:26Z', updated_by=None), KnowledgeBaseResponse(knowledge_base_id='4999b45d-6176-4060-ae5e-7700583abe77', project_id='95de11d3-a7f6-4ab1-bc61-59180e62b0b4', user_id='0b9b68df-88dc-4401-b0cb-fbc967aea677', name='TESTE_PAGINACAO_8', description='TESTE_PAGINACAO_TESTE_PAGINACAO_TESTE_PAGINACAO_TESTE_PAGINACAO_TESTE_PAGINACAO_', access_level=<AccessLevelType.PROTECTED: 'PROTECTED'>, embedding_m

In [12]:
import os
from uuid import uuid4

from dotenv import load_dotenv
from iaragenai import IaraGenAI

load_dotenv()


def main():
    client = IaraGenAI(
        client_id=os.getenv("IARA_CLIENT_ID", "your_client_id"),
        client_secret=os.getenv("IARA_CLIENT_SECRET", "your_client_secret"),
        environment=os.getenv("IARA_ENVIRONMENT", "prod"),
        access_token=os.getenv("IARA_ACCESS_TOKEN", "your_project_key_access"),
        correlation_id=str(uuid4()),
    )

    knowledge_base_id = os.getenv("KNOWLEDGE_BASE_ID", "112dc826-d2b7-4e70-9573-4420ab000bc0")

    kb = client.knowledge_base.get(knowledge_base_id=knowledge_base_id)

    print(kb)


if __name__ == "__main__":
    main()

knowledge_base_id='112dc826-d2b7-4e70-9573-4420ab000bc0' project_id='95de11d3-a7f6-4ab1-bc61-59180e62b0b4' user_id='0b9b68df-88dc-4401-b0cb-fbc967aea677' name='KB_Catchs_Auditoria' description='KB de catchs da auditoria. Criado para homologação do agente do Multiagente RO' access_level=<AccessLevelType.PROTECTED: 'PROTECTED'> embedding_model=EmbeddingModel(provider_name='azure_openai', model_name='text-embedding-ada-002', model_dimension=1536) created_at='2026-03-27T20:36:26Z' updated_at='2026-03-27T20:36:26Z' updated_by=None


In [13]:

@mcp.tool()
def consultar_manual_metodologia(consulta: str) -> str:
    """
    Busca conteúdo na Knowledge Base de metodologia via IARA.
    Retorna trechos relevantes formatados para uso em análise por LLM.
    """

    log_message("INFO", "=" * 80)
    log_message("INFO", f"🔧 TOOL: consultar_manual_metodologia | Consulta: {consulta[:80]}")
    log_message("INFO", "=" * 80)

    try:
        if not consulta or not consulta.strip():
            return "❌ Parâmetro 'consulta' é obrigatório."

        from dotenv import load_dotenv
        import os

        env_path = os.path.join(os.path.dirname(__file__), ".env")
        if os.path.exists(env_path):
            load_dotenv(env_path)

        client_id = os.getenv("CLIENT_ID")
        client_secret = os.getenv("CLIENT_SECRET")
        access_key = os.getenv("IARA_ACCESS_KEY") or os.getenv("ACCESS_KEY")
        environment = os.getenv("ENVIRONMENT", "homol")
        provider = os.getenv("PROVIDER", "azure_openai")
        knowledge_base_id = os.getenv(
            "KB_METODOLOGIA_ID"
        )

        if not all([client_id, client_secret, access_key]):
            return "❌ Credenciais IARA não configuradas."

        os.environ.setdefault("HTTP_PROXY", "http://proxynew.itau:8080")
        os.environ.setdefault("HTTPS_PROXY", "http://proxynew.itau:8443")

        from iaragenai import IaraGenAI
        from iaragenai.apis.resources.datafoundation_api.types import (
            SimilaritySearchKnowledgeBaseVersionReference
        )

        kb_ref = SimilaritySearchKnowledgeBaseVersionReference(
            knowledge_base_id=knowledge_base_id,
            knowledge_base_version=None
        )

        client = IaraGenAI(
            client_id=client_id,
            client_secret=client_secret,
            access_token=access_key,
            environment=environment,
            provider=provider
        )

        response = client.similarity_search.search(
            text=consulta.strip(),
            top_k=5,
            strategy="cosine_similarity",
            knowledge_bases=[kb_ref],
            rerank=True
        )

        # ✅ AQUI ESTÁ A CORREÇÃO CRÍTICA
        if not isinstance(response, list) or not response:
            return f"⚠️ Nenhum resultado encontrado para: {consulta}"

        lines = [
            "📚 RESULTADOS DA KNOWLEDGE BASE – METODOLOGIA",
            "",
            f"Consulta: {consulta}",
            f"KB: {knowledge_base_id}",
            f"Resultados retornados: {len(response)}",
            ""
        ]

        for idx, item in enumerate(response, start=1):
            text = (getattr(item, "text", "") or "").strip()
            score = getattr(item, "score", None)
            rerank_score = getattr(item, "reranker_score", None)

            document = getattr(item, "document", None)
            doc_name = getattr(document, "document_name", None) if document else None

            # limite duro de conteúdo para não quebrar agentes downstream
            if len(text) > 600:
                text = text[:600] + " [...]"

            lines.append(f"[{idx}] Score: {score:.3f} | Rerank: {rerank_score:.3f}")
            if doc_name:
                lines.append(f"Documento: {doc_name}")
            if text:
                lines.append(f"Trecho relevante: {text}")
            lines.append("")

        log_message("SUCCESS", f"Busca KB retornou {len(response)} resultado(s)")
        return "\n".join(lines)

    except Exception as e:
        import traceback
        log_message("ERROR", str(e))
        log_message("ERROR", traceback.format_exc())
        return f"❌ Erro ao consultar metodologia: {e}"
 

NameError: name 'mcp' is not defined

In [14]:
import os
from uuid import uuid4

from dotenv import load_dotenv
from iaragenai import IaraGenAI
from iaragenai.apis.resources.datafoundation_api.types import (
    SimilaritySearchKnowledgeBaseVersionReference,
)

load_dotenv()

# ── Configuração ────────────────────────────────────────────────
KNOWLEDGE_BASE_ID = os.getenv("KNOWLEDGE_BASE_ID", "2715c6f3-0d35-4c57-a90f-57bdea65e23a")
MODEL = os.getenv("IARA_MODEL", "gpt-4o-mini")
PROVIDER = os.getenv("IARA_PROVIDER", "azure_openai")
TOP_K = 5

# 👇 sua pergunta sobre o KB
pergunta = "Oque tem na sua base de conhecimento do KB"

# ── 1) Busca trechos relevantes no KB ───────────────────────────
search_client = IaraGenAI(
    client_id=os.getenv("IARA_CLIENT_ID"),
    client_secret=os.getenv("IARA_CLIENT_SECRET"),
    environment=os.getenv("IARA_ENVIRONMENT", "homol"),
    access_token=os.getenv("IARA_ACCESS_TOKEN"),
    correlation_id=str(uuid4()),
)

kb_ref = SimilaritySearchKnowledgeBaseVersionReference(
    knowledge_base_id=KNOWLEDGE_BASE_ID,
    knowledge_base_version=None,
)

trechos = search_client.similarity_search.search(
    text=pergunta,
    top_k=TOP_K,
    strategy="cosine_similarity",
    knowledge_bases=[kb_ref],
)

# ── 2) Monta contexto com os trechos ────────────────────────────
contexto_blocos = []
for idx, item in enumerate(trechos or [], start=1):
    texto = (getattr(item, "text", "") or "").strip()
    doc = getattr(item, "document", None)
    doc_name = getattr(doc, "document_name", None) if doc else None
    cabecalho = f"[{idx}] {doc_name}" if doc_name else f"[{idx}]"
    contexto_blocos.append(f"{cabecalho}\n{texto}")

contexto = "\n\n".join(contexto_blocos) if contexto_blocos else "(sem resultados)"

print(f"📚 {len(trechos or [])} trecho(s) recuperado(s) do KB\n")

# ── 3) Pergunta ao LLM usando os trechos como contexto ──────────
chat_client = IaraGenAI(
    client_id=os.getenv("IARA_CLIENT_ID"),
    client_secret=os.getenv("IARA_CLIENT_SECRET"),
    environment=os.getenv("IARA_ENVIRONMENT", "homol"),
    provider=PROVIDER,
    correlation_id=str(uuid4()),
)

system_prompt = (
    "você diz tudo sobre Homologacao-KB-Multiagente_RO"
)

user_prompt = f"Pergunta: {pergunta}\n\nTrechos da KB:\n{contexto}"

response = chat_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0.2,
)

print("🤖 Resposta:\n")
print(response.choices[0].message.content)


📚 5 trecho(s) recuperado(s) do KB

🤖 Resposta:

A base de conhecimento (KB) que você forneceu contém informações detalhadas sobre diversos **CATCHes** (mecanismos de controle ou testes automatizados) relacionados a processos de avaliação de risco, conformidade e governança, especialmente no contexto de **KYP (Conheça Seu Parceiro)** e **KYC (Conheça Seu Cliente)**. Abaixo está um resumo do que está incluído na KB:

---

### **1. Estrutura Geral da KB**
A KB é composta por descrições de **CATCHes**, suas respectivas avaliações de apontamentos, inputs, contexto de entrada, hipóteses, algoritmos e outputs. Cada CATCH é identificado por um código único (ex.: H03745, H00224, H01364, H00420) e está associado a um objetivo específico de controle ou análise.

---

### **2. Exemplos de CATCHes e Seus Objetivos**
#### **CATCH H03745 – Corretora operando sem aprovação de KYP**
- **Objetivo:** Verificar se corretoras utilizadas em Fundos Intrag passaram pela avaliação de KYP.
- **Apontamento:** Nã